# RepairScope-X — Notebook 21
## MDLM: constant denoising effort per block + model-native repair

**Research question:** Notebook 20 gave 128/256-token generations the same nominal 64 forward evaluations as the 64-token arm, reducing denoising steps available to each 32-token block. Does holding the **configured refinement budget at 32 steps per 32-token block** improve answer correctness, and can MDLM repair its own newly generated failures?

**Two sequential stages, independently checkpointed on Google Drive:**

1. **Controlled generation:** the identical 24 frozen problems at 128 and 256 tokens, with respectively 128 and 256 total *requested* sampler steps (32 per block). Run in **12 batches of four**; preserve existing Notebook 20 results as a separate, historical **fixed-total-64-step** reference. No GPU rerun of old results.
2. **Native self-repair:** 8 failed 128-token generations selected before any repair. Four actions (`tail25`, `tail50`, `middle25`, `global`) × 2 seeds, giving **64 trial records** on eight independent failed problems. Run in **16 batches of four**; frozen observed tokens are audited at the token level.

The same pinned MDLM checkpoint, Qwen2 custom model source, original prompt template and official `MDLMSampler` are used. The model is not retrained.

**Experimental cautions:** The repair masks differ across scopes, so their actual model-forward counts are reported rather than claimed to be matched. The two seeds are repeats on the same eight failures; independent statistical units are failed problems. Numeric word-problem grading is provisional, and reasoning correctness requires further review.

**Execution:** fresh Colab T4 runtime → run preparation cells in order → 12 generation batches → freeze failure cohort → 16 repair batches → export. Resume by reconnecting Drive, rerunning preparation/model-loading cells, and skipping successful trials. Never reset/delete the persistent Drive output folder when resuming.
**Version 2 checkpoint fix:** Section 11 now tolerates only scientifically identical, audited GPU timing changes. No generation or repair protocol changed.


### 0. Safe Python library setup — first code cell

In [ ]:
# Must be first code cell; does not import NumPy into the notebook kernel.
import os, sys, subprocess, pathlib
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'  # public model snapshots
if sys.version_info < (3,10):
    raise RuntimeError('Python 3.10 or newer is required.')
check="import numpy, torch; assert numpy.asarray([1,2]).sum()==3; print('NumPy',numpy.__version__,'Torch',torch.__version__)"
r=subprocess.run([sys.executable,'-c',check],capture_output=True,text=True)
if r.returncode:
    print(r.stderr[-7000:])
    raise RuntimeError('Scientific Python environment already corrupted. Colab: Runtime > Disconnect and delete runtime; reconnect; do NOT reinstall NumPy inside a live runtime.')
print(r.stdout)
# Narrow, pinned Python interfaces only; never reinstall PyTorch, NumPy, SciPy or Pandas.
packages=['transformers==4.57.0','accelerate==1.11.0',
          'huggingface-hub==0.35.3','tokenizers==0.22.1','safetensors==0.6.2']
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','--no-deps',*packages],check=True)
preflight="import torch,transformers,accelerate,huggingface_hub,tokenizers,safetensors; from transformers import AutoTokenizer; from transformers.processing_utils import Unpack; from transformers.utils import TransformersKwargs; from transformers.modeling_attn_mask_utils import _prepare_4d_attention_mask; print('model interface',transformers.__version__)"
r=subprocess.run([sys.executable,'-c',preflight],capture_output=True,text=True)
if r.returncode:
    print(r.stderr[-9000:])
    raise RuntimeError('Model library import preflight failed. Start from a fresh runtime and report this error rather than running GPU stages.')
print(r.stdout)

### 1. Colab runtime + exact source revision + persistent checkpoint folder

In [ ]:
import base64, zlib, json, re, math, ast as pyast, gc, time, random, traceback, hashlib, zipfile, csv, shutil, subprocess, sys, importlib.util, os
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import torch, transformers
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
assert torch.cuda.is_available(), 'Choose T4 GPU and restart from the beginning.'
assert transformers.__version__=='4.57.0', 'Transformers version mismatch; restart with a fresh runtime.'
GPU=torch.cuda.get_device_name(0); VRAM=torch.cuda.get_device_properties(0).total_memory/2**30
assert VRAM >= 7, f'Insufficient GPU memory: {VRAM:.2f} GB'
print('GPU',GPU,'VRAM',round(VRAM,2),'GB','torch',torch.__version__)
from google.colab import drive
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive/RepairScopeX/RSX21_MDLMCostControlled_v1')
DRIVE.mkdir(parents=True,exist_ok=True)
# Verify persistent destination is writable; do not fall back to ephemeral /content.
(DRIVE/'drive_write_test.txt').write_text('ok',encoding='utf-8')
assert (DRIVE/'drive_write_test.txt').read_text()=='ok'
SOURCE_COMMIT='ca176752fbceec49c6b4777a2c18ae88e4eb10ed'
ROOT=Path('/content/rsx_v6_official_repo')
if not (ROOT/'.git').exists():
    ROOT.mkdir(parents=True,exist_ok=True)
    subprocess.run(['git','init','-q',str(ROOT)],check=True)
    subprocess.run(['git','-C',str(ROOT),'remote','add','origin','https://github.com/ZHZisZZ/dllm.git'],check=True)
try:
    sha=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],stderr=subprocess.DEVNULL,text=True).strip()
except subprocess.CalledProcessError:
    sha=''
if sha!=SOURCE_COMMIT:
    for retry in range(3):
        try:
            subprocess.run(['git','-C',str(ROOT),'fetch','--quiet','--depth','1','origin',SOURCE_COMMIT],check=True,timeout=240)
            subprocess.run(['git','-C',str(ROOT),'checkout','-q','--force','FETCH_HEAD'],check=True,timeout=80)
            break
        except Exception:
            if retry==2: raise
            time.sleep(2**retry)
sha=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
assert sha==SOURCE_COMMIT
print('Official code SHA',sha)

### 2. Load byte-identical official sampler source with inference-only package initializers

In [ ]:
# Exact official sampler files, copied BYTE-FOR-BYTE from the pinned upstream commit.
SHIM=Path('/content/rsx_v6_inference_only')
files=[
    'dllm/core/samplers/__init__.py','dllm/core/samplers/base.py',
    'dllm/core/samplers/mdlm.py','dllm/core/samplers/bd3lm.py',
    'dllm/core/samplers/utils.py','dllm/core/schedulers/__init__.py',
    'dllm/core/schedulers/alpha.py','dllm/core/schedulers/kappa.py',
]
for rel in files:
    source=ROOT/rel; destination=SHIM/rel
    assert source.exists(),source
    destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,destination)
    assert hashlib.sha256(source.read_bytes()).digest()==hashlib.sha256(destination.read_bytes()).digest()
for rel in ['dllm/__init__.py','dllm/core/__init__.py']:
    f=SHIM/rel;f.parent.mkdir(parents=True,exist_ok=True)
    f.write_text('# Inference namespace, without unrelated training imports.\n')
# Model class from exact official source, no checkpoint `trust_remote_code` needed.
MODEL_FILE=ROOT/'dllm/pipelines/a2d/models/qwen2/modeling_qwen2.py'
assert MODEL_FILE.is_file()
sys.path.insert(0,str(SHIM))
from dllm.core.samplers import MDLMSampler, MDLMSamplerConfig, BD3LMSampler, BD3LMSamplerConfig
model_module_name='rsx_v6_a2d_qwen2_source'
if model_module_name not in sys.modules:
    spec=importlib.util.spec_from_file_location(model_module_name,str(MODEL_FILE))
    module=importlib.util.module_from_spec(spec)
    sys.modules[model_module_name]=module
    spec.loader.exec_module(module)
else:
    module=sys.modules[model_module_name]
A2DQwen2Config=module.A2DQwen2Config
A2DQwen2LMHeadModel=module.A2DQwen2LMHeadModel
assert A2DQwen2Config.model_type=='a2d-qwen2'
print('Official samplers:',MDLMSampler.__name__,BD3LMSampler.__name__)

### 3. Frozen 24-problem benchmark (identical to NB20)

In [ ]:
PAYLOAD_SHA="cc04b82051f62f8640b92c418571d5efd3fd89334bf4be126bfae6f508ff5328"
_encoded="eNq1WW2P27gR/iuEsUW+eBeWLPkNRYttkyAX3KXFbooULQofLdMWYUlUScqOcrj/3mdIyZad7a1tnIFssmtyhjPPPPO2+aVnVKUTYXqzf//SK7VaZCKfy2Vv1ktUVdil2hWzhGdyobmVqpgFk0E8iwa9fm/JLTfCdm/OV1qYFGcl11bSfTo9SNOJVnlJQv8wgtlUsKLKF0IbFk77DH94sWQTJr7yxGY1U0WCW4pBalklXsByvRaWBeEDexK20gXjbIs3loxradNcWJlAQQlTDN50GgW+djhkP7//4dPjj/PHT89f3j3N2B/963/6+QGmecW9GTQPfu2fj8bk9wcjiPosGHg0wuFZcNwMjcvAiMY3oEafDT0W8TlQDAe3wgKaLwIjvAE1kCPBuGHG4Bw44vBmeJDqiwAJbsCOYNTJlbNKxzC6GUGiy/CIb5IsYXQJGvHN0IgvRGMY/P5wDMGMsGFHcFa+RDdLl+jS8jG9AT2IH/El9eM+HN4KEFL9PSJrk082M6u5LGbxcHQMgjs8z/WfuLUs5QZWl4IXlWVlxgvr7sFBbpk0LB4M2ErAzUW9//6BsYCZ/1ZcC7ZSyjK1ajQYlvCC5XwjcJmtNc9N5xByP6yYtEBuIwzBe3qDIPbSp0dsUVkrtOcp26y/PzEiywzs0ewORE7VjuVVkrKlwlOIllO7gu8shVsHT//cLU9BGA9eQXw0GsfXIv6R5zBmUdWGjZi9N6nU1pscD/4Aj1ZA6HMqaiY4LE+UseyOuhP7sPcG5IIzJa+7do9eM3ocDMdXG63Sgi1cxIIQRiWKQM95wrVShel7Y3dCrlNZrBE5pCEG5gf2waVJAWOTDUhWMAHKZE2mUfysC0ujh8mCRfBvtRJaFIiqRibj3TVe0IIva4fTUmRyK3T9wL6QaihYaClWyNBM8C1MJJ0baRM6pXyVycZ9VqaqEH0X+0xam5GgwoFmz1ZsBZywXhhvS/AaXlp86xgIydbcg1WIypcmR1wxUBa+ORRsn3zxKPRbQS1yBIIA+g7BbiSj+LWEH4ejayP5meygT1ujeE4lk37K4WPN/so177OPvBDWV8CPQusaNQIAwc9oFs9GD5TDB49PVViibytwN0YlhTFJlXHri2amjHHveP30FOphlvn8lCuvIFH5QhZOQupWs6LMYUrzYi2ME6eUpzs5QyAmLoV8lKqakC61TES3nk5fze3o6tx+hkFauXpq5FdmJaU6fsh5UbMlr8AL6i90jJg7FGFpUmlie1Y7QZtqIdx53xcwkk1SmS1xaV/JCN0uZ8LgVc6Mr87+90g6gnTFcwkrtagK3OizvwuqudK0WAcjVoIJS5cwCwTsHjkA05ON8I0v5dnKd5WGMjhPeb6o4ISmbPog3hi2VqSsCXWIyCMVD3pT9JqlWhNQ/CVpnhl1rMIgtzVoauTSFQfesM0lamtTNxHaJ9qiSxE4GIBut/QKPMuOmser7AriyfVF2Igybbq1Fist4Te3SiNP2Y5TLFLh/iFYEZZUVUb4JKbKm4nEIhuY2oqmt6MImYo4Wqod9VXbqb3QTAX8829kOe43NqEX+R7mq6fQ65rh7SVNDL7yHaz11ZJI7vOjg73vagdN3qyuoq6b+yp07Bmutw7tlMaUdUczi6TBDIbbtM/sDjWB7f6f/V729DWn97SSHSHzqvUYM9d8CaKYo18ewLzXkjeKwquTF7UykeZNm71En5BVmyoDbHAoYutKwhbTEIUlQls82unRW6kySfXqHXX5RtQpipgB8MW6PfOq3NHo5MgrOZU66iOHF5tzQpbyGDcOBh+Z25jmOtOgFesUzvbc1c3voTgpoi+EwWx5XjZhSFAr7qej4VEo3IXzQvG4Ih6BTVScOPFK25rlgme+4+0o71C0jPDs+kuVbHAVNQ7Eyjj6GKWjRko/AVGR0byjC9DLpWIUw1sjkWE0geduVXlRC3OjPfipBeCpAA6vXSQm4RTztCoRXKfJaX5JAS0vslHCMaBOYtJhDtVyb8hv62l3jOXxKDueROdEIjj5TckFkXCMLJVffUCcVUa56uZPaRuUGa020eDo0PH0odsVMB3SPeskqMhkWdeVYHAWpybBtZ5gzdHAt6QBvJ2IFILvuuYaVbY4nHkL3XDEzYZ2OJ9+ja+8SUPkXxDvpY4kDp6/orqLQXwOBOPB1RC8o2WAOMSeavQ5Uwrq1CNqf4gaOObShHJOFOtMmtQv9i+cY1YpUKU7AXYw+ovt6OX0v6jWTyTC33xB79GQfw4ow8HVDD9QGAzpbFRKbTxLRp1Pc7WVh1C/YYnm3zA3YcbB30mqVIYPjcCW5Ut2rSq3j7Eg2u9wXgVpRtcEKXA43c/i7tVjUP1nDrJG1m0MreqLCRRPo2uxeltp2bSZR73hheHekc/iKw3kghqbK6i48MS/Kb3gtFH63Z2ZVJXwaynYHQuROIva7SR+D24X/IejRdFt9cDGbc3NlSNyDM+qGtH0Wo9/rDY0i0hqHkFEY7ukPRcEcCZpGnbJQO4cbxeVMuM1CcRjf8P4K2i6W+lvdiuj09j8rsIk6lAh6OJR2w2n4VmtNxpd6++/4BWnIgFzhqhtlUnvq9JHeQi6Y09vat26zlmScUO/hnJ98S3fQogE40mzi7TSC2zwkzHSHLf3Ohyjv/n3TgvJXtJd2ouQ8r2F6mifIFz+0+8ZN+LCqTlmA2AznATTfvPfknOuUV+2Am4+iZJL/ZyoUvxz/klZQUkWBPO3yKh8/rddMX/PZVZpYeb4qjLw8pskCHNhU0WYr+RXRPj5w+N9GI+wAxSbPpugzGvW4v7r/wDPufiP"
_raw=zlib.decompress(base64.b64decode(_encoded))
assert hashlib.sha256(_raw).hexdigest()==PAYLOAD_SHA
SOURCES=json.loads(_raw.decode('utf-8'))['sources']
assert len(SOURCES)==24 and len(set(x['problem_id'] for x in SOURCES))==24
assert Counter(x['dataset'] for x in SOURCES)=={'countdown_fresh':8,'gsm8k':8,'svamp':8}
print('Frozen sources:',Counter(x['dataset'] for x in SOURCES))

### 4. Task-aware answer verifier and regression tests

In [ ]:
# Shared benchmark evaluator. Countdown requires a legal expression with all numbers once.
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
PROMPT_RE=re.compile(r'Use the numbers\s+(.+?)\s+exactly once\s+to produce the target\s+([-+]?(?:\d+(?:\.\d*)?|\.\d+))',re.I)
ARITH_RE=re.compile(r'[0-9eE+\-*/().=\s]{3,}')

def equals(a,b):
    return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-8)

def lastnum(text):
    x=NUM_RE.findall(str(text).replace(',',''))
    try:return float(x[-1]) if x else None
    except:return None

def eval_arith(expr):
    try:tree=pyast.parse(expr,mode='eval')
    except (SyntaxError,ValueError):return None
    numbers=[]
    def visit(node):
        if isinstance(node,pyast.Expression):return visit(node.body)
        if isinstance(node,pyast.Constant) and type(node.value) in (int,float):
            v=float(node.value)
            if not math.isfinite(v):raise ValueError('nonfinite')
            numbers.append(v);return v
        if isinstance(node,pyast.BinOp) and type(node.op) in (pyast.Add,pyast.Sub,pyast.Mult,pyast.Div):
            a,b=visit(node.left),visit(node.right)
            if isinstance(node.op,pyast.Add):return a+b
            if isinstance(node.op,pyast.Sub):return a-b
            if isinstance(node.op,pyast.Mult):return a*b
            if b==0:raise ZeroDivisionError()
            return a/b
        if isinstance(node,pyast.UnaryOp) and type(node.op) in (pyast.UAdd,pyast.USub):
            a=visit(node.operand);return -a if isinstance(node.op,pyast.USub) else a
        raise ValueError('forbidden AST')
    try:
        ans=visit(tree)
        return (ans,numbers) if math.isfinite(ans) else None
    except (ZeroDivisionError,ValueError,OverflowError):return None

def countdown_check(text,prompt):
    m=PROMPT_RE.search(prompt)
    if not m: raise ValueError('Unexpected Countdown prompt: '+prompt[:200])
    required=[float(x) for x in NUM_RE.findall(m.group(1))];goal=float(m.group(2))
    normalized=str(text).replace('×','*').replace('÷','/').replace('\\times','*')
    for chunk in ARITH_RE.findall(normalized):
        if not any(k in chunk for k in '+-*/'):continue
        for candidate in chunk.split('='):
            candidate=candidate.strip();value=eval_arith(candidate)
            if value is None:continue
            output,nums=value
            if len(nums)==len(required) and all(equals(a,b) for a,b in zip(sorted(nums),sorted(required))) and equals(output,goal):
                return True,candidate
    return False,None

def verify(text,s):
    strict_matches=FINAL_RE.findall(str(text))
    strict=float(strict_matches[-1]) if strict_matches else None
    numeric=lastnum(text)
    if s['dataset']=='countdown_fresh': success,expr=countdown_check(text,s['prompt'])
    else: success=equals(numeric,s['target']);expr=None
    return {'task_success':bool(success),'strict_success':equals(strict,s['target']),
            'numeric_success':equals(numeric,s['target']),'strict_prediction':strict,
            'numeric_prediction':numeric,'valid_expression':expr}

_demo='Use the numbers 19, 29, and 24 exactly once to produce the target 14.'
assert countdown_check('19 + 24 - 29 = 14',_demo)[0]
assert not countdown_check('29 - 19 + 24 = 14',_demo)[0]
assert not countdown_check('29-19=10 FINAL_ANSWER:14',_demo)[0]
print('Verification regression tests PASS')

### 5. The **actual** earlier Notebook-20 MDLM reference records (64 total-step settings)

In [ ]:
REFERENCE_SHA256 = "aa06e0a452dffb1b3d8755c132acd9188b1faa3d9ab53548d996478cd795a7f1"
_ref_bytes=zlib.decompress(base64.b64decode("""..."""))
assert hashlib.sha256(_ref_bytes).hexdigest()==REFERENCE_SHA256
UPSTREAM=json.loads(_ref_bytes.decode('utf-8'))
REFERENCE=UPSTREAM['records']
assert len(REFERENCE)==48
assert Counter(x['budget'] for x in REFERENCE)=={128:24,256:24}
assert len(set(x['problem_id'] for x in REFERENCE if x['budget']==128))==24
assert all(x['status']=='ok' and int(x['nfe'])>0 for x in REFERENCE)
assert all(x['problem_id'] in {s['problem_id'] for s in SOURCES} for x in REFERENCE)
REFERENCE_LOOKUP={(x['problem_id'],int(x['budget'])):x for x in REFERENCE}
print('Reference provenance:',UPSTREAM['manifest'].get('config',{}).get('protocol','Notebook20'))
print('Earlier fixed-64-step generations:',len(REFERENCE),'and embedded SHA:',REFERENCE_SHA256)

### 6. Verified model loader, deterministic protocol, restart-safe record operations

In [ ]:
PROTOCOL='rsx21-mdlm-per-block32-plus-selfrepair-v1'
MODELS={
    'MDLM': dict(
        repo='dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-mdlm-v0.1',
        sha='a284e895a6248256baf2f60502e54aba61b24c1a',
        actions=['tail25','tail50','middle25','global'],
        sampler_cls=MDLMSampler,config_cls=MDLMSamplerConfig,
    ),
    'BD3LM': dict(
        repo='dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-bd3lm-v0.1',
        sha='4bcf3a00ba6ec28d9cf9c3ebc4a5a5dc01c6023c',
        actions=['tail25','tail50','tail75','global'],
        sampler_cls=BD3LMSampler,config_cls=BD3LMSamplerConfig,
    ),
}
OUT_LEN=128; BLOCK=32; STEPS=128; TEMPERATURE=.15
GEN_SEED=38191; REPAIR_SEEDS=[38192,38193]; SELECT_MAX=8
PROMPT_SUFFIX='\nReturn a concise answer. For word problems finish with FINAL_ANSWER: <number>. For Countdown give a valid arithmetic expression using each supplied number once.'
CTX=None
SRC_MAP={s['problem_id']:s for s in SOURCES}
assert len(SRC_MAP)==24

def normalized_json(x):
    if isinstance(x,Path):return str(x)
    if isinstance(x,torch.Tensor):return x.detach().cpu().tolist()
    if hasattr(x,'item'):
        try:return x.item()
        except Exception:pass
    raise TypeError('Nonserializable '+repr(type(x)))

def atomic_json(path,value):
    path=Path(path);p=path.with_suffix(path.suffix+'.tmp')
    with p.open('w',encoding='utf-8') as f:
        json.dump(value,f,indent=2,ensure_ascii=False,default=normalized_json,allow_nan=False)
        f.flush();os.fsync(f.fileno())
    os.replace(p,path)

def append_record(path,row):
    with open(path,'a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False,default=normalized_json,allow_nan=False)+'\n')
        f.flush();os.fsync(f.fileno())

def completed_map(path,signature):
    records={}
    if not Path(path).exists():return records
    with open(path,encoding='utf-8') as f:
        for lineno,line in enumerate(f,1):
            if not line.strip():continue
            try:r=json.loads(line)
            except Exception as ex:raise ValueError(f'Malformed checkpoint JSONL {path}:{lineno}; backup before editing') from ex
            if r['signature']!=signature:raise ValueError('Checkpoint run-signature mismatch: '+str(path))
            if r['id'] in records:raise ValueError('Duplicate trial ID in checkpoint: '+str(r['id']))
            records[r['id']]=r
    return records

def model_dir(family):
    f=DRIVE/family
    f.mkdir(parents=True,exist_ok=True)
    return f

def config_signature(family):
    assert family=='MDLM', 'NB21 runs MDLM only.'
    m=MODELS[family]
    frozen=dict(protocol=PROTOCOL,family=family,model_id=m['repo'],model_sha=m['sha'],
                source_commit=SOURCE_COMMIT,source_payload_sha=PAYLOAD_SHA,
                n_sources=len(SOURCES),out_tokens=OUT_LEN,block=BLOCK,configured_steps_per_block=32,
                temperature=TEMPERATURE,gen_seed=GEN_SEED,repair_seeds=REPAIR_SEEDS,
                select_max=SELECT_MAX,actions=['tail25','tail50','middle25','global'],budgets=[128,256],fixed64_reference_sha=REFERENCE_SHA256)
    return hashlib.sha256(json.dumps(frozen,sort_keys=True).encode()).hexdigest(),frozen

def files_for(family):
    folder=model_dir(family)
    return {'folder':folder,'manifest':folder/'run_manifest.json',
            'natural':folder/'generation_trials.jsonl',
            'repair':folder/'repair_trials.jsonl',
            'cohort':folder/'frozen_failure_ids.json',
            'errors':folder/'errors.jsonl'}

def init_family(family):
    paths=files_for(family);sig,cfg=config_signature(family)
    if paths['manifest'].exists():
        old=json.loads(paths['manifest'].read_text())
        if old['signature']!=sig:raise RuntimeError('Different protocol found for '+family+'; do not overwrite existing run. Rename Drive folder or restore previous notebook.')
    else:
        atomic_json(paths['manifest'],dict(signature=sig,config=cfg,
            created_utc=datetime.now(timezone.utc).isoformat(),gpu=GPU,torch_version=torch.__version__,transformers_version=transformers.__version__))
    return paths,sig

def unload_model():
    global CTX
    if CTX is not None:
        print('Unloading',CTX['family'])
        CTX['sampler']=None
        CTX['model']=None
        CTX['tokenizer']=None
        CTX=None
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()

def load_family(family):
    global CTX
    assert family in MODELS
    if CTX and CTX['family']==family:
        print(family,'already loaded');return
    unload_model()
    m=MODELS[family]; paths,sig=init_family(family)
    print('Loading',family,'public Hugging Face checkpoint (no token needed)',m['repo'])
    # HF cache-first; otherwise pinned full-snapshot download; simple retry.
    try:
        local=snapshot_download(repo_id=m['repo'],revision=m['sha'],local_files_only=True)
    except Exception:
        for trial in range(3):
            try:
                local=snapshot_download(repo_id=m['repo'],revision=m['sha'],max_workers=4)
                break
            except Exception:
                if trial==2:raise
                time.sleep(2**trial)
    local=Path(local)
    assert (local/'config.json').exists() and (local/'model.safetensors').exists()
    try:
        tokenizer=AutoTokenizer.from_pretrained(str(local),local_files_only=True,trust_remote_code=False,use_fast=True)
        if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
        if tokenizer.bos_token_id is None:tokenizer.bos_token=tokenizer.pad_token
        if tokenizer.mask_token_id is None:
            tokenizer.mask_token='<|mask|>'
        mask_id=int(tokenizer.mask_token_id)
        pad_id=int(tokenizer.pad_token_id)
        eos_id=int(tokenizer.eos_token_id)
        config=A2DQwen2Config.from_pretrained(str(local),local_files_only=True)
        if config.model_type!='a2d-qwen2':raise ValueError('Wrong checkpoint architecture: '+str(config.model_type))
        model=A2DQwen2LMHeadModel.from_pretrained(
            str(local),config=config,dtype=torch.float16,
            local_files_only=True,low_cpu_mem_usage=True,attn_implementation='sdpa',
        ).to('cuda').eval()
        nrows=int(model.get_input_embeddings().weight.shape[0])
        assert 0<=mask_id<nrows and mask_id not in [pad_id,eos_id],('Mask token missing from trained vocabulary',mask_id,nrows)
        sampler=m['sampler_cls'](model=model,tokenizer=tokenizer)
        CTX=dict(family=family,model=model,tokenizer=tokenizer,sampler=sampler,
                 mask=mask_id,eos=eos_id,pad=pad_id,paths=paths,signature=sig,
                 model_sha=m['sha'],model_id=m['repo'],classes=m)
        print('MODEL LOADED',family,'GPU allocated GB',round(torch.cuda.memory_allocated()/2**30,3),'mask id',mask_id)
    except Exception as exc:
        append_record(paths['errors'],dict(stage='model_load',signature=sig,error=repr(exc),traceback=traceback.format_exc(),timestamp=datetime.now(timezone.utc).isoformat()))
        raise

def prepare_ids(s):
    t=CTX['tokenizer']
    msg=[{'role':'system','content':'You solve quantitative reasoning tasks carefully.'},
         {'role':'user','content':s['prompt']+PROMPT_SUFFIX}]
    ids=t.apply_chat_template(msg,tokenize=True,add_generation_prompt=True,enable_thinking=False)
    if isinstance(ids,dict):ids=ids['input_ids']
    ids=[int(i) for i in ids]
    assert ids and CTX['mask'] not in ids
    return ids

def visible_answer_tokens(token_ids):
    result=[]
    for i in token_ids:
        if int(i) in [CTX['eos'],CTX['pad']]:break
        result.append(int(i))
    return result

def decode(token_ids):
    return CTX['tokenizer'].decode([int(i) for i in token_ids],skip_special_tokens=True).strip()

def sampler_config(answer_budget):
    # Fixed 32 steps for EACH 32-token block, NOT a fixed overall NFE.
    requested_steps=32*math.ceil(int(answer_budget)/BLOCK)
    a=dict(max_new_tokens=int(answer_budget),steps=requested_steps,block_size=BLOCK,
           temperature=TEMPERATURE,remasking='low_confidence',return_dict=True,
           right_shift_logits=False)
    if CTX['family']=='BD3LM':a['steps_per_block']=32
    return CTX['classes']['config_cls'](**a)

class CountCalls:
    def __init__(self,module):self.module=module;self.count=0;self.h=None
    def cb(self,*args):self.count+=1
    def __enter__(self):self.count=0;self.h=self.module.register_forward_pre_hook(self.cb);return self
    def __exit__(self,*args):self.h.remove();self.h=None

def set_seed(seed):
    torch.manual_seed(int(seed));torch.cuda.manual_seed_all(int(seed));random.seed(int(seed))

@torch.no_grad()
def real_generate(prompt,answer_budget,seed):
    set_seed(seed)
    t=time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    with CountCalls(CTX['model']) as calls:
        result=CTX['sampler'].sample([list(prompt)],sampler_config(answer_budget),return_dict=True)
    raw=result.sequences[0].detach().cpu().tolist()
    if CTX['family']=='MDLM':
        assert raw[:len(prompt)]==list(prompt), 'MDLM changed observed prompt'
        answer=raw[len(prompt):]
    else:
        offset=math.ceil(len(prompt)/BLOCK)*BLOCK
        assert raw[offset-len(prompt):offset]==list(prompt),'BD3LM changed observed prompt'
        answer=raw[offset:]
    assert not any(t==CTX['mask'] for t in answer), 'Unresolved generation mask(s)'
    assert len(answer)<=answer_budget
    return dict(token_ids=visible_answer_tokens(answer),nfe=int(calls.count),
                raw_generated_tokens=len(answer),wall_seconds=float(time.perf_counter()-t),
                peak_vram_gb=float(torch.cuda.max_memory_allocated()/2**30))

### 7. Explicit MDLM CUDA smoke test (generation + native infill)

In [ ]:
load_family('MDLM')
source=SOURCES[0]
prompt=prepare_ids(source)
smoke=real_generate(prompt,32,45001)
assert len(smoke['token_ids'])>=4 and smoke['nfe']>0
# A real infill must leave observed prompt and selected answer tokens unchanged.
original=smoke['token_ids']; old=list(original);canvas=[CTX['mask'] if k>=len(old)//2 else old[k] for k in range(len(old))]
steps_infill=32*math.ceil((len(prompt)+len(canvas))/BLOCK)
cfg=MDLMSamplerConfig(steps=steps_infill,block_size=BLOCK,temperature=.15,remasking='low_confidence',return_dict=True)
with CountCalls(CTX['model']) as call_audit:
    result=CTX['sampler'].infill([prompt+canvas],cfg,return_dict=True)
seq=result.sequences[0].detach().cpu().tolist()
assert seq[:len(prompt)]==prompt
assert seq[len(prompt):len(prompt)+len(old)//2]==old[:len(old)//2]
assert CTX['mask'] not in seq
print('REAL CUDA generation + native infill SMOKE PASS. Forward calls:',smoke['nfe'],call_audit.count)

### 8. Cost-controlled generation code and persistent batch runner

In [ ]:
BUDGETS=(128,256)
assert CTX is not None and CTX['family']=='MDLM'

def generation_trial(s,budget):
    ret=real_generate(prepare_ids(s),int(budget),GEN_SEED)
    decoded=decode(ret['token_ids'])
    v=verify(decoded,s)
    return dict(id=f"{s['problem_id']}|fixed32x32|{budget}|{GEN_SEED}",problem_id=s['problem_id'],
       dataset=s['dataset'],target=float(s['target']),prompt=s['prompt'],budget=int(budget),
       generation_seed=GEN_SEED,output=decoded,verification=v,
       task_success=bool(v['task_success']),strict_success=bool(v['strict_success']),
       numeric_success=bool(v['numeric_success']), visible_answer_tokens=len(ret['token_ids']),
       step_policy='32_per_32token_block',configured_total_steps=32*math.ceil(budget/BLOCK),
       budget_exhaustion_proxy=bool(ret['raw_generated_tokens']==budget and len(ret['token_ids'])>=budget-2),**ret)

def gen_progress():
    done=completed_map(CTX['paths']['natural'],CTX['signature'])
    for budget in BUDGETS:
        ready=sum(int(x['budget']==budget and x['status']=='ok') for x in done.values())
        print(f'Controlled generation {budget}: {ready}/24 completed')

def generation_batch(budget,start,n=4,max_minutes=8):
    assert CTX is not None and CTX['family']=='MDLM'
    assert int(budget) in BUDGETS and 0<=start<24 and 1<=n<=4 and start+n<=24
    path=CTX['paths']['natural'];sig=CTX['signature'];done=completed_map(path,sig)
    began=time.monotonic()
    for s in SOURCES[start:start+n]:
        tid=f"{s['problem_id']}|fixed32x32|{budget}|{GEN_SEED}"
        if tid in done:
            assert done[tid]['status']=='ok'
            print('SKIP',tid);continue
        if time.monotonic()-began>max_minutes*60:
            print('Reached soft cutoff. Rerun this batch to resume.');break
        try:
            r=generation_trial(s,budget);r.update(signature=sig,status='ok')
            append_record(path,r)
            print('SAVED',tid,'task=',r['task_success'],'NFEs=',r['nfe'])
        except Exception as exc:
            append_record(CTX['paths']['errors'],dict(stage='generation',id=tid,error=repr(exc),
                  traceback=traceback.format_exc(),utc=datetime.now(timezone.utc).isoformat()))
            raise RuntimeError('Generation failed. Existing Drive checkpoints are safe. Check errors.jsonl.') from exc
    gen_progress()

### 9. 128-token MDLM: 6 short batches × 4 generations

In [ ]:
generation_batch(128,0,n=4,max_minutes=8)

In [ ]:
generation_batch(128,4,n=4,max_minutes=8)

In [ ]:
generation_batch(128,8,n=4,max_minutes=8)

In [ ]:
generation_batch(128,12,n=4,max_minutes=8)

In [ ]:
generation_batch(128,16,n=4,max_minutes=8)

In [ ]:
generation_batch(128,20,n=4,max_minutes=8)

### 10. 256-token MDLM: 6 short batches × 4 generations

In [ ]:
generation_batch(256,0,n=4,max_minutes=8)

In [ ]:
generation_batch(256,4,n=4,max_minutes=8)

In [ ]:
generation_batch(256,8,n=4,max_minutes=8)

In [ ]:
generation_batch(256,12,n=4,max_minutes=8)

In [ ]:
generation_batch(256,16,n=4,max_minutes=8)

In [ ]:
generation_batch(256,20,n=4,max_minutes=8)

### 10a. Safe intermediate download: 48 generation results (before any repairs)

In [ ]:
# Export the entire generation stage separately before proceeding to repairs.
assert CTX and CTX['family']=='MDLM'
_checkpoint=completed_map(CTX['paths']['natural'],CTX['signature'])
_expected={f"{s['problem_id']}|fixed32x32|{b}|{GEN_SEED}" for s in SOURCES for b in BUDGETS}
assert set(_checkpoint)==_expected, f'Generation stage incomplete: {len(_expected-set(_checkpoint))} missing'
assert all(x['status']=='ok' for x in _checkpoint.values())
_export_zip=Path('/content/RepairScopeX_NB21_MDLM_ConstantPerBlock_Generation_only.zip')
with zipfile.ZipFile(_export_zip,'w',compression=zipfile.ZIP_DEFLATED) as _z:
    _z.write(CTX['paths']['manifest'],CTX['paths']['manifest'].name)
    _z.write(CTX['paths']['natural'],CTX['paths']['natural'].name)
print('Generation-only backup verified:',_export_zip)
from google.colab import files
files.download(str(_export_zip))

### 11. Freeze or safely resume eight 128-token failures (v2 metadata-only fix)

This replaces the raw generation-file checksum check with a scientifically pinned 48-record fingerprint. It accepts only the two previously audited timing-only ZIP variants, and otherwise refuses inconsistent frozen cohorts.


In [ ]:
# NB21 v2: Cohort selection using a content fingerprint, not GPU timing.
# The original and the new generation-only ZIPs were compared field-by-field.
# Only wall_seconds changed. This fingerprint pins every scientific field of all 48 records.
from collections import defaultdict, Counter

# Audited against BOTH supplied Notebook-21 generation archives.
AUDITED_SCIENTIFIC_SHA256 = "8c6abe099a1c5f9f8932f44f35f0929c28545398a607e626e90a7474ec29cd90"
AUDITED_LEGACY_FILE_SHA256 = {
    "1c9a2e85c99622bf35c1813fabc481d3241d9d94f9608c55b4603262a94fcc39",
    "cc8c0996c1c53dc57e840808a1217d2890ba127642b587d02395f1d2a9dbd658",
}

def scientific_generation_digest(rows):
    # Exclude only execution telemetry, which has no bearing on outputs or cohort.
    normalized = [{k: v for k, v in rec.items()
                   if k not in ('wall_seconds', 'peak_vram_gb')}
                  for rec in rows]
    normalized.sort(key=lambda rec: rec['id'])
    body = json.dumps(normalized, sort_keys=True, ensure_ascii=False,
                      separators=(',', ':'), allow_nan=False).encode('utf-8')
    return hashlib.sha256(body).hexdigest()

def choose_failures():
    assert CTX is not None and CTX['family'] == 'MDLM'
    p = CTX['paths']; sig = CTX['signature']
    gens = completed_map(p['natural'], sig)
    expected = {f"{s['problem_id']}|fixed32x32|{b}|{GEN_SEED}"
                for s in SOURCES for b in BUDGETS}
    assert set(gens) == expected, f"Generation checkpoint not complete: {len(expected-set(gens))} missing"
    assert len(gens) == 48
    assert all(x['status'] == 'ok' and x['signature'] == sig for x in gens.values())
    digest = scientific_generation_digest(list(gens.values()))
    if digest != AUDITED_SCIENTIFIC_SHA256:
        raise RuntimeError('Scientific generation records differ from the two audited NB21 ZIPs. '
                           'Do not reuse a frozen cohort: investigate changes in predictions or labels.')

    pool = defaultdict(list)
    for x in gens.values():
        if x['budget'] == 128 and not x['task_success'] and len(x['token_ids']) >= 16:
            # Revalidate the failure before selecting it.
            assert not verify(x['output'], SRC_MAP[x['problem_id']])['task_success']
            pool[x['dataset']].append(x)
    quotas = [('countdown_fresh', 3), ('gsm8k', 3), ('svamp', 2)]
    assert all(len(pool[k]) >= n for k,n in quotas), 'Not enough eligible failures for predeclared dataset quotas.'
    selected = []
    for k,n in quotas:
        ordered = sorted(pool[k], key=lambda x:hashlib.sha256(
            ('rsx21cohort:'+x['problem_id']).encode()).hexdigest())
        selected.extend(ordered[:n])
    ids=[x['problem_id'] for x in selected]
    assert len(ids)==len(set(ids))==8
    new_content = {
        'source_generation_sha256': hashlib.sha256(p['natural'].read_bytes()).hexdigest(),
        'generation_signature': sig,
        'ids': ids,
        'budget': 128,
        'dataset_counts': dict(Counter(x['dataset'] for x in selected)),
        'scientific_generation_sha256': digest,
    }

    if p['cohort'].exists():
        current = json.loads(p['cohort'].read_text())
        fields = ('generation_signature','ids','budget','dataset_counts')
        for field in fields:
            if current.get(field) != new_content[field]:
                raise RuntimeError(f"Frozen cohort changed in {field}; existing checkpoint preserved.")
        prev_science = current.get('scientific_generation_sha256')
        if prev_science is not None and prev_science != digest:
            raise RuntimeError('Previously frozen scientific content differs; checkpoint preserved.')
        prior_file_sha = current.get('source_generation_sha256')
        if prior_file_sha != new_content['source_generation_sha256']:
            # Only accept metadata-only cases covered by the TWO source archives audited
            # before constructing this notebook. Do not silently accept any other hash.
            if prior_file_sha not in AUDITED_LEGACY_FILE_SHA256:
                raise RuntimeError('Stored frozen cohort refers to an unreviewed generation file checksum. '
                                   'Supply that older checkpoint for an explicit comparison; nothing was overwritten.')
            print('Accepted previously audited timing-only generation change; frozen cohort is unchanged.')
        # Never overwrite a frozen cohort record to make an error disappear.
    else:
        atomic_json(p['cohort'], new_content)
        print('Created new frozen failure cohort (scientific fingerprint verified).')

    # Important if resuming after repairs have already begun.
    repairs=completed_map(p['repair'],sig)
    allowed={f"{f['problem_id']}|{a}|{seed}" for f in selected
             for a in ('tail25','tail50','middle25','global')
             for seed in REPAIR_SEEDS}
    if set(repairs)-allowed:
        raise RuntimeError('Saved repair trials refer to a different failure cohort. Stop and inspect.')
    selected_by_id={f['problem_id']:f for f in selected}
    for rid,repair in repairs.items():
        failure=selected_by_id[repair['problem_id']]
        if repair.get('original_output') != failure['output']:
            raise RuntimeError(f'Repair origin mismatch for {rid}; existing repairs preserved.')
    print('FROZEN: 8 genuine native failures',new_content['dataset_counts'],
          '| generation SHA256:',digest[:16], '| existing repairs:',len(repairs))
    return selected

SELECTED_FAILURES=choose_failures()


### 12. Native MDLM masked-token repair, preservation audit and compute accounting

In [ ]:
ACTIONS=('tail25','tail50','middle25','global')
REPAIR_SEEDS=(38192,38193)

def repair_canvas(old,action,mask):
    n=len(old)
    assert n>=16
    if action=='global': changed=set(range(n))
    elif action=='tail25': changed=set(range(max(1,int(.75*n)),n))
    elif action=='tail50': changed=set(range(max(1,int(.50*n)),n))
    elif action=='middle25':
        width=max(1,math.ceil(.25*n));start=(n-width)//2
        changed=set(range(start,start+width))
    else: raise ValueError(action)
    assert len(changed)>0
    locked=set(range(n))-changed
    return [mask if k in changed else int(old[k]) for k in range(n)],sorted(locked), sorted(changed)

@torch.no_grad()
def self_repair(failure,action,seed):
    s=SRC_MAP[failure['problem_id']]
    old=[int(t) for t in failure['token_ids']]
    prompt=prepare_ids(s)
    canvas,locked,changed=repair_canvas(old,action,CTX['mask'])
    assert all(canvas[i]==old[i] for i in locked)
    assert all(canvas[i]==CTX['mask'] for i in changed)
    # MDLM.infIll divides steps over ALL blocks, including observed prompt blocks.
    # Give every 32-token input block 32 requested denoising steps; empty blocks skip.
    total_steps=32*math.ceil((len(prompt)+len(canvas))/BLOCK)
    cfg=MDLMSamplerConfig(steps=total_steps,block_size=BLOCK,temperature=TEMPERATURE,
        remasking='low_confidence',return_dict=True,right_shift_logits=False)
    set_seed(seed);ts=time.perf_counter();torch.cuda.reset_peak_memory_stats()
    with CountCalls(CTX['model']) as calls:
        out=CTX['sampler'].infill([prompt+canvas],cfg,return_dict=True)
    seq=[int(t) for t in out.sequences[0].detach().cpu().tolist()]
    assert seq[:len(prompt)]==prompt,'Observed question altered by MDLM infill.'
    repaired=seq[len(prompt):]
    assert len(repaired)==len(old), 'Repaired canvas length changed'
    assert all(repaired[i]==old[i] for i in locked),'Frozen tokens were modified'
    assert CTX['mask'] not in repaired,'Unfilled masks remain'
    shown=visible_answer_tokens(repaired)
    text=decode(shown)
    v=verify(text,s)
    original_visible=visible_answer_tokens(old)
    changes=sum(int(a!=b) for a,b in zip(repaired,old))
    preserve=1.0-changes/len(old)
    same_numeric=lastnum(text)==lastnum(failure['output']) if lastnum(text) is not None else False
    old_task_ok=verify(failure['output'],s)['task_success']
    assert not old_task_ok,'Starting point must be a task failure.'
    return dict(id=f"{failure['problem_id']}|{action}|{seed}",problem_id=failure['problem_id'],
         dataset=s['dataset'],action=action,seed=int(seed),original_output=failure['output'],
         repaired_output=text,original_token_count=len(old),num_reopened=len(changed),num_locked=len(locked),
         preservation=float(preserve),locked_tokens_exact=True,changed_token_positions=changes,
         wrong_answer_recurrence_numeric=bool(not v['task_success'] and same_numeric),
         exact_wrong_output_recurrence=bool(not v['task_success'] and repaired==old),
         task_success=bool(v['task_success']),strict_success=bool(v['strict_success']),
         numeric_success=bool(v['numeric_success']),verification=v,
         native_method='MDLMSampler.infill',requested_steps=int(total_steps),nfe=int(calls.count),
         wall_seconds=float(time.perf_counter()-ts),peak_vram_gb=float(torch.cuda.max_memory_allocated()/2**30),
         raw_repaired_tokens=repaired)

REPAIR_JOBS=[(s,act,seed) for s in SELECTED_FAILURES for act in ACTIONS for seed in REPAIR_SEEDS]
assert len(REPAIR_JOBS)==64

def repair_progress():
    d=completed_map(CTX['paths']['repair'],CTX['signature'])
    print('Saved repair trials:',len(d),'/64')

def repair_batch(start,n=4,max_minutes=8):
    assert CTX is not None and CTX['family']=='MDLM'
    assert 0<=start<64 and 1<=n<=4 and start+n<=64
    jobs=REPAIR_JOBS[start:start+n];path=CTX['paths']['repair'];sig=CTX['signature']
    done=completed_map(path,sig)
    began=time.monotonic()
    for f,action,seed in jobs:
        tid=f"{f['problem_id']}|{action}|{seed}"
        if tid in done:
            assert done[tid]['status']=='ok'; print('SKIP',tid);continue
        if time.monotonic()-began>max_minutes*60:
            print('Soft 8-minute cutoff reached. Rerun cell to complete this batch.');break
        try:
            row=self_repair(f,action,seed);row.update(signature=sig,status='ok')
            append_record(path,row)
            print('SAVED',tid,'success=',row['task_success'],'NFEs=',row['nfe'])
        except Exception as exc:
            append_record(CTX['paths']['errors'],dict(stage='self_repair',id=tid,error=repr(exc),
                traceback=traceback.format_exc(),utc=datetime.now(timezone.utc).isoformat()))
            raise RuntimeError('Self-repair failed, previous trials safe on Drive. Check errors.jsonl.') from exc
    repair_progress()

### 13. Repairs 01–32 — eight checkpointed 4-trial batches

In [ ]:
repair_batch(0,n=4,max_minutes=8)

In [ ]:
repair_batch(4,n=4,max_minutes=8)

In [ ]:
repair_batch(8,n=4,max_minutes=8)

In [ ]:
repair_batch(12,n=4,max_minutes=8)

In [ ]:
repair_batch(16,n=4,max_minutes=8)

In [ ]:
repair_batch(20,n=4,max_minutes=8)

In [ ]:
repair_batch(24,n=4,max_minutes=8)

In [ ]:
repair_batch(28,n=4,max_minutes=8)

### 14. Repairs 33–64 — eight checkpointed 4-trial batches

In [ ]:
repair_batch(32,n=4,max_minutes=8)

In [ ]:
repair_batch(36,n=4,max_minutes=8)

In [ ]:
repair_batch(40,n=4,max_minutes=8)

In [ ]:
repair_batch(44,n=4,max_minutes=8)

In [ ]:
repair_batch(48,n=4,max_minutes=8)

In [ ]:
repair_batch(52,n=4,max_minutes=8)

In [ ]:
repair_batch(56,n=4,max_minutes=8)

In [ ]:
repair_batch(60,n=4,max_minutes=8)

### 15. Integrity checks, paired length comparisons, clustered repair summaries

In [ ]:
import csv

def write_csv(file,rows):
    if not rows:return
    keys=list(rows[0].keys())
    with open(file,'w',newline='',encoding='utf-8') as h:
        w=csv.DictWriter(h,fieldnames=keys,extrasaction='ignore');w.writeheader();w.writerows(rows)

def wilson95(k,n):
    z=1.95996398454;p=k/n;scale=1+z*z/n
    a=(p+z*z/(2*n))/scale;b=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/scale
    return max(0,a-b),min(1,a+b)

def paired_exact(a,b):
    n01=sum((not bool(x)) and bool(y) for x,y in zip(a,b))
    n10=sum(bool(x) and (not bool(y)) for x,y in zip(a,b))
    discord=n01+n10
    p=1.0 if discord==0 else min(1.0,2*sum(math.comb(discord,k) for k in range(min(n01,n10)+1))/2**discord)
    return {'new_only':n01,'old_only':n10,'discordant':discord,'two_sided_exact_p':p}

def finish_experiment():
    assert CTX and CTX['family']=='MDLM'
    p=CTX['paths'];sig=CTX['signature'];g=completed_map(p['natural'],sig);r=completed_map(p['repair'],sig)
    expected_g={f"{s['problem_id']}|fixed32x32|{b}|{GEN_SEED}" for s in SOURCES for b in BUDGETS}
    expected_r={f"{s['problem_id']}|{act}|{seed}" for s in SELECTED_FAILURES for act in ACTIONS for seed in REPAIR_SEEDS}
    assert set(g)==expected_g and len(g)==48, f'Incomplete generations: missing {len(expected_g-set(g))}'
    assert set(r)==expected_r and len(r)==64, f'Incomplete repairs: missing {len(expected_r-set(r))}'
    assert all(x['status']=='ok' for x in list(g.values())+list(r.values()))
    assert all(x['locked_tokens_exact'] for x in r.values())
    generations=sorted(g.values(),key=lambda x:(x['budget'],x['problem_id']))
    repairs=sorted(r.values(),key=lambda x:(x['problem_id'],x['action'],x['seed']))
    generation_summary=[];paired=[]
    ids=[s['problem_id'] for s in SOURCES]
    for budget in BUDGETS:
        new={x['problem_id']:x for x in generations if x['budget']==budget}
        old={x['problem_id']:x for x in REFERENCE if x['budget']==budget}
        assert set(new)==set(old)==set(ids)
        nh=sum(bool(x['task_success']) for x in new.values());oh=sum(bool(x['task_success']) for x in old.values())
        lo,hi=wilson95(nh,24)
        generation_summary.append(dict(budget=budget,conditions='fixed32_per_block_vs_fixed_total64',n=24,
             new_successes=nh,old_successes=oh,new_task_rate=nh/24,old_task_rate=oh/24,
             wilson95_low=lo,wilson95_high=hi,new_mean_nfe=sum(x['nfe'] for x in new.values())/24,
             old_mean_nfe=sum(x['nfe'] for x in old.values())/24,
             new_mean_seconds=sum(x['wall_seconds'] for x in new.values())/24,
             old_mean_seconds=sum(x['wall_seconds'] for x in old.values())/24))
        paired.append(dict(budget=budget,**paired_exact([old[i]['task_success'] for i in ids],
                                            [new[i]['task_success'] for i in ids])))
    repair_summary=[]
    for act in ACTIONS:
        group=[x for x in repairs if x['action']==act]
        assert len(group)==16
        success=sum(int(x['task_success']) for x in group)
        # Cluster 2 repeated trials by original failure, report problem-level observed success.
        best_by_problem={pid:any(x['task_success'] for x in group if x['problem_id']==pid) for pid in {x['problem_id'] for x in group}}
        p_success=sum(best_by_problem.values());lo,hi=wilson95(p_success,8)
        repair_summary.append(dict(action=act,trials=len(group),unique_failed_problems=8,successful_trials=success,
            trial_success_rate=success/16,problems_ever_repaired=p_success,
            problem_ever_repair_rate=p_success/8,wilson95_problem_low=lo,wilson95_problem_high=hi,
            mean_nfe=sum(x['nfe'] for x in group)/len(group),
            mean_preservation=sum(x['preservation'] for x in group)/len(group),
            mean_wall_seconds=sum(x['wall_seconds'] for x in group)/len(group),
            same_wrong_numeric_given_failure=sum(int(x['wrong_answer_recurrence_numeric']) for x in group if not x['task_success'])/
                 max(1,sum(int(not x['task_success']) for x in group))))
    out=p['folder']
    write_csv(out/'controlled_length_summary.csv',generation_summary)
    write_csv(out/'paired_fixed_total_vs_per_block.csv',paired)
    write_csv(out/'repair_action_summary.csv',repair_summary)
    oracle=sum(int(any(x['task_success'] for x in repairs if x['problem_id']==sid)) for sid in [x['problem_id'] for x in SELECTED_FAILURES])/8
    best_fixed=max(x['trial_success_rate'] for x in repair_summary)
    gate=dict(run_signature=sig,expected_natural_generations=48,actual_natural_generations=len(g),
         expected_repair_trials=64,actual_repair_trials=len(r),
         unique_repair_problems=8,all_frozen_positions_audited=True,
         native_method='official_mdlm_infill',generation_policy='32_requested_steps_per_32_token_block',
         positive_repair_trials=sum(int(x['task_success']) for x in repairs),
         best_fixed_trial_success=best_fixed,in_sample_oracle_any_seed_success=oracle,
         signal_ready=bool(sum(int(x['task_success']) for x in repairs)>0),complete=True)
    atomic_json(out/'NB21_GATE.json',gate)
    artifacts=[p['manifest'],p['natural'],p['repair'],p['cohort'],out/'controlled_length_summary.csv',
      out/'paired_fixed_total_vs_per_block.csv',out/'repair_action_summary.csv',out/'NB21_GATE.json']
    hashes={x.name:hashlib.sha256(x.read_bytes()).hexdigest() for x in artifacts}
    atomic_json(out/'artifact_manifest.json',dict(sha256=hashes,upstream_reference_sha256=REFERENCE_SHA256))
    artifacts.append(out/'artifact_manifest.json')
    zip_file=Path('/content/RepairScopeX_NB21_MDLM_ConstantPerBlock_Repair_results.zip')
    with zipfile.ZipFile(zip_file,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in artifacts:z.write(f,f.name)
    print('NB21 COMPLETE. Gate:',json.dumps(gate,indent=2))
    print('Generation:',generation_summary, chr(10), 'Repair:',repair_summary, chr(10), 'ZIP:',zip_file)
    return zip_file

RESULT_ZIP=finish_experiment()
from google.colab import files
files.download(str(RESULT_ZIP))

### 16. Next step after running

Return `RepairScopeX_NB21_MDLM_ConstantPerBlock_Repair_results.zip` for analysis. If there are still no repairs, do not repeat this checkpoint again without changing the research question. Finish the pending BD3LM length experiment, then compare models on the original paired problems with an explicit compute policy.

A signal gate of `false` is a valid negative finding; it is not a reason to discard the run. This notebook reports verified token-preservation but does not claim to have verified the semantic correctness of all word-problem reasoning chains.